# 10 — Services Layer
Explore the Protocol interfaces, mock vs real switching, and factory functions.

In [ ]:
import sys; sys.path.insert(0, '/home/claude/codebase/code/src')
import os; os.environ['ENABLE_MOCK']='true'; os.environ['REDIS_ENABLED']='false'

## Four service protocols

In [ ]:
from services.base import IDataService, ITicketService, IMetadataService, IVectorService
print('Service protocols:')
for svc in [IDataService, ITicketService, IMetadataService, IVectorService]:
    print(f'  {svc.__name__}: {[m for m in dir(svc) if not m.startswith("_")]}')

## MockDatabricksService

In [ ]:
from services.databricks.mock import MockDatabricksService

svc = MockDatabricksService()
print('=== Normal mode ===')
rows = svc.query('SELECT * FROM analytics.retention_metrics WHERE period = \'2024-Q1\'')
for row in rows:
    print(f"  GRR: {row['gross_retention_rate']}%  Churn: {row['churn_rate']}%")

print('\n=== Low GRR mode ===')
low_svc = MockDatabricksService(low_grr=True)
rows = low_svc.query('SELECT * FROM analytics.retention_metrics WHERE period = \'2024-Q1\'')
for row in rows:
    print(f"  GRR: {row['gross_retention_rate']}%  At-risk accounts: {row['at_risk_accounts']}")

## MockJiraService

In [ ]:
from services.jira.mock import MockJiraService

svc = MockJiraService()
print('Open issues:')
for issue in svc.search_issues('project = DGC', max_results=5):
    fields = issue['fields']
    print(f"  {issue['key']}: {fields['summary']} [{fields['status']['name']}]")

# Create a ticket
ticket = svc.create_issue(
    summary='[Auto] GRR dropped below 85%',
    description='GRR at 78% — needs immediate review.',
    issue_type='Bug',
    priority='High',
    labels=['data-quality', 'retention']
)
print(f'\nCreated: {ticket["key"]} — {ticket["fields"]["summary"]}')
print(f'Labels: {ticket["fields"]["labels"]}')
print(f'Inspectable tickets list: {len(svc.tickets)} ticket(s)')

## MockCollibraService

In [ ]:
from services.collibra.mock import MockCollibraService

svc = MockCollibraService()
assets = svc.search_assets('retention')
print('Assets for retention:')
for a in assets:
    dq = svc.get_data_quality(a['id'])
    print(f"  {a['id']}: {a['name']} (owner: {a['owner']})")
    print(f"    DQ score: {dq['score']}% ({dq['passed']}/{dq['total_rules']} rules passed)")

## NullVectorService (mock pgvector)

In [ ]:
from services.pgvector.mock import NullVectorService
from langchain_core.documents import Document

svc = NullVectorService()

# Similarity search — keyword scoring against canned governance docs
results = svc.similarity_search('gross retention rate definition', k=3)
print(f'Results for "gross retention rate definition":')
for doc, score in results:
    print(f'  Score: {score:.2f} | Topic: {doc.metadata.get("topic")}')
    print(f'    {doc.page_content[:80]}...')

## Factory — ENABLE_MOCK switching

In [ ]:
from services.factory import get_data_service, get_ticket_service, get_metadata_service, get_vector_service
from services.databricks.mock import MockDatabricksService
from services.jira.mock import MockJiraService
from services.collibra.mock import MockCollibraService
from services.pgvector.mock import NullVectorService

os.environ['ENABLE_MOCK'] = 'true'
print('ENABLE_MOCK=true:')
print('  data:    ', type(get_data_service()).__name__)
print('  tickets: ', type(get_ticket_service()).__name__)
print('  metadata:', type(get_metadata_service()).__name__)
print('  vector:  ', type(get_vector_service()).__name__)

## Protocol conformance check

In [ ]:
from services.base import IDataService, ITicketService, IMetadataService, IVectorService
from services.databricks.mock import MockDatabricksService
from services.jira.mock import MockJiraService
from services.collibra.mock import MockCollibraService
from services.pgvector.mock import NullVectorService

checks = [
    (MockDatabricksService(), IDataService),
    (MockJiraService(), ITicketService),
    (MockCollibraService(), IMetadataService),
    (NullVectorService(), IVectorService),
]
print('Protocol conformance:')
for svc, protocol in checks:
    conforms = isinstance(svc, protocol)
    status = '✅' if conforms else '❌'
    print(f'  {status} {type(svc).__name__} implements {protocol.__name__}')